# Marginal Distribution of Features

Juan José H. Beltrán. Ph.D. in Computer Sciences.

In [1]:
# Start working from the root directory of the repository
import os
os.chdir('../')

In [2]:
import pandas as pd

## Import Dataset

In [3]:
df = pd.read_csv('data/obesity.csv')
df.head()

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,Female,21.0,1.62,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight
1,Female,21.0,1.52,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23.0,1.80,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight
3,Male,27.0,1.80,87.0,no,no,3.0,3.0,Sometimes,no,2.0,no,2.0,0.0,Frequently,Walking,Overweight_Level_I
4,Male,22.0,1.78,89.8,no,no,2.0,1.0,Sometimes,no,2.0,no,0.0,0.0,Sometimes,Public_Transportation,Overweight_Level_II


### Target Variable Grouping

In [5]:
def replace_categories(row):
    if row['NObeyesdad'] == 'Overweight_Level_I' or row['NObeyesdad'] == 'Overweight_Level_II':
        return 'Overweight'
    elif row['NObeyesdad'] == 'Obesity_Type_I' or row['NObeyesdad'] == 'Obesity_Type_II' or row['NObeyesdad'] == 'Obesity_Type_III':
        return 'Obesity'
    else:
        return row['NObeyesdad']

df['NObeyesdad'] = df.apply(replace_categories, axis=1)

### Overview

In [6]:
df.describe()

,Age,Height,Weight,FCVC,NCP,CH2O,FAF,TUE
count,2111.000000,2111.000000,2111.000000,2111.000000,2111.000000,2111.000000,2111.000000,2111.000000
mean,24.312600,1.701677,86.586058,2.419043,2.685628,2.008011,1.010298,0.657866
std,6.345968,0.093305,26.191172,0.533927,0.778039,0.612953,0.850592,0.608927
min,14.000000,1.450000,39.000000,1.000000,1.000000,1.000000,0.000000,0.000000
25%,19.947192,1.630000,65.473343,2.000000,2.658738,1.584812,0.124505,0.000000
50%,22.777890,1.700499,83.000000,2.385502,3.000000,2.000000,1.000000,0.625350
75%,26.000000,1.768464,107.430682,3.000000,3.000000,2.477420,1.666678,1.000000
max,61.000000,1.980000,173.000000,3.000000,4.000000,3.000000,3.000000,2.000000


In [7]:
target = 'NObeyesdad'
features = df.columns[df.columns != target]
print(f'The dataset has {len(features)} features and {len(df)} samples.')

The dataset has 16 features and 2111 samples.


### Make a Profile Report for Exploration

In [7]:
from data_profiling import ProfileReport

profile_report = ProfileReport(df, title="Obesity Dataset Profiling Report")
profile_report.to_file("figures/obesity-report.html")

c:\Users\juanj\OneDrive\Documentos\GitHub\tc6002-statistical-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Export report to file: 100%|██████████| 1/1 [00:00<00:00, 68.34it/s]


## Compute Marginal Distributions

In [8]:
# Categorize features into categorical and numerical
categorical_features = list(df.select_dtypes(include=['object']).columns)
categorical_features.remove(target)

continuous_features = list(df.select_dtypes(include=['float64', 'int64']).columns)

In [9]:
from scripts.marginal_distributions import marginal_distributions

res = marginal_distributions(
    df, categorical_features, continuous_features,
    out_dir="figures/marginal-distributions",
    category_orders={"CAEC": ["no", "Sometimes", "Frequently", "Always"],
                     "CALC": ["no", "Sometimes", "Frequently", "Always"]},
)
print(res["support"])

                                       type  n_unique  \
feature                                                 
Gender                               binary         2   
family_history_with_overweight       binary         2   
FAVC                                 binary         2   
CAEC                            categorical         4   
SMOKE                                binary         2   
SCC                                  binary         2   
CALC                            categorical         4   
MTRANS                          categorical         5   
Age                              continuous      1402   
Height                           continuous      1574   
Weight                           continuous      1525   
FCVC                             continuous       810   
NCP                              continuous       635   
CH2O                             continuous      1268   
FAF                              continuous      1190   
TUE                            

## Analyze Continuous Features

In [10]:
from scripts.continuous_features_analysis import analyze_continuous_features
out = analyze_continuous_features(df, continuous_features, target=target)

In [11]:
out

{'conditional_stats':    feature                level    n        mean  mean_ci_low  mean_ci_high  \
 0      Age  Insufficient_Weight  272   19.783237    19.474836     20.113856   
 1      Age        Normal_Weight  287   21.738676    21.163676     22.365854   
 2      Age           Overweight  580   25.207328    24.600384     25.802058   
 3      Age              Obesity  972   25.806181    25.466332     26.178549   
 4   Height  Insufficient_Weight  272    1.691117     1.679392      1.702715   
 5   Height        Normal_Weight  287    1.676585     1.665920      1.687495   
 6   Height           Overweight  580    1.695792     1.688338      1.703730   
 7   Height              Obesity  972    1.715553     1.710438      1.721272   
 8   Weight  Insufficient_Weight  272   49.906330    49.224034     50.631157   
 9   Weight        Normal_Weight  287   62.155052    61.110322     63.146594   
 10  Weight           Overweight  580   78.176050    77.428284     78.968081   
 11  Weight        

## Analyze Categorical Features

In [13]:
categorical_features

['Gender',
 'family_history_with_overweight',
 'FAVC',
 'CAEC',
 'SMOKE',
 'SCC',
 'CALC',
 'MTRANS']

In [9]:
from scripts.categorical_features_analysis import analyze_categorical_features
out_cat_features = analyze_categorical_features(
    df, categorical_features, target=target,
    out_dir="figures/categorical-features",
    category_orders={"CAEC": ["no", "Sometimes", "Frequently", "Always"],
                     "CALC": ["no", "Sometimes", "Frequently", "Always"]})